# ChromaDB — Similarity Search Project

### What this does
Creates a ChromaDB collection of grocery items, adds them with auto-generated embeddings, then performs similarity search using a query term — returning the top 3 most semantically similar items.

### Full Code

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Embedding function using sentence-transformers
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# ChromaDB client
client = chromadb.Client()
collection_name = "my_grocery_collection"

def main():
    try:
        # Create collection with cosine distance metric
        collection = client.create_collection(
            name=collection_name,
            metadata={"description": "A collection for storing grocery data"},
            configuration={
                "hnsw": {"space": "cosine"},
                "embedding_function": ef
            }
        )
        print(f"Collection created: {collection.name}")

        # Sample documents
        texts = [
            'fresh red apples', 'organic bananas', 'ripe mangoes',
            'whole wheat bread', 'farm-fresh eggs', 'natural yogurt',
            'frozen vegetables', 'grass-fed beef', 'free-range chicken',
            'fresh salmon fillet', 'aromatic coffee beans', 'pure honey',
            'golden apple', 'red fruit'
        ]

        # Auto-generate IDs: food_1, food_2, ...
        ids = [f"food_{index + 1}" for index, _ in enumerate(texts)]

        # Add documents — ChromaDB auto-generates embeddings
        collection.add(
            documents=texts,
            metadatas=[{"source": "grocery_store", "category": "food"} for _ in texts],
            ids=ids
        )

        # Verify documents stored
        all_items = collection.get()
        print(f"Number of documents: {len(all_items['documents'])}")

        # Run similarity search
        perform_similarity_search(collection, all_items)

    except Exception as error:
        print(f"Error: {error}")


def perform_similarity_search(collection, all_items):
    try:
        query_term = "apple"

        results = collection.query(
            query_texts=[query_term],
            n_results=3
        )

        # Handle no results
        if not results or not results['ids'] or len(results['ids'][0]) == 0:
            print(f'No documents found similar to "{query_term}"')
            return

        # Display top 3 results
        print(f'Top 3 similar documents to "{query_term}":')
        for i in range(min(3, len(results['ids'][0]))):
            doc_id = results['ids'][0][i]
            score = results['distances'][0][i]
            text = results['documents'][0][i]
            if not text:
                print(f' - ID: {doc_id}, Text: "Text not available", Score: {score:.4f}')
            else:
                print(f' - ID: {doc_id}, Text: "{text}", Score: {score:.4f}')

    except Exception as error:
        print(f"Error in similarity search: {error}")


if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection created: my_grocery_collection
Number of documents: 14
Top 3 similar documents to "apple":
 - ID: food_13, Text: "golden apple", Score: 0.3825
 - ID: food_1, Text: "fresh red apples", Score: 0.4809
 - ID: food_14, Text: "red fruit", Score: 0.5965


**Note on scores:** since we're using **cosine distance** (not similarity), **lower score = more similar** — opposite of what you might expect. `golden apple` scoring 0.38 means it's the closest match to the query `"apple"`.